# R4b — 학습 데이터 정답을 고쳐서 다시 학습하면 재현율이 오르는가

## 왜 하나

R4에서 dev-L·cal-L 정답을 LLM으로 재검증한 결과, 원래 '멀쩡한' 응답의 40%가 표시되지 않은 환각을 포함했고 엄격 재현율 상한(약 0.50)의 상당 부분이 정답 누락이었다(B 문장 끝 0.501 → 0.621, 위약을 뺀 순이득 +0.063).
그런데 보정 정답에서는 **문장 끝(0.621)이 4토큰 앞부분 판정(0.48~0.54)보다 크게 높았고**, 판정기들의 완성 문장 AUROC는 오히려 떨어졌다(B 0.857 → 0.810).
판정기들이 **같은 정답 누락이 있는 학습 데이터**(gold-large 3,000응답)로 학습돼, 누락된 종류의 환각(근거 없는 꾸밈말, 리뷰 내용 왜곡, 미묘한 사실 오류)을 "멀쩡함"으로 배웠기 때문으로 본다.

## 무엇을 하나

1. **학습 데이터 재판정**: `claude-sonnet-5-5`(배치, 약 $10)가 gold-large 3,000응답의 모든 문장을 R4와 같은 지시문으로 재판정한다. 공개 LLM 2개(Qwen 14B·72B)는 Opus와의 일치도 관문을 넘지 못해 쓰지 않는다.
   관문 G는 R4에서 받은 Sonnet 판정의 cal-L 부분으로 계산한다(비용 없음). Sonnet은 평가 정답(Opus)과 같은 계열이라 R4b-2(원래 정답 비열등)를 함께 본다.
2. **보정 학습 정답**: 원래 정답 ∪ 재판정 H (R4 C1과 같은 규칙, 원래 정답은 지우지 않음). 추가 문장의 시작 토큰은 재판정 구절 위치.
3. **재학습**: R1b **V0b와 완전히 같은 방식**(student_vark에서 시작, 문장 단위 정답, 모든 판정 지점, 지점별 이진 분류, 1에폭, 시드 42·7·2026)으로 학습 정답만 바꾼다.
   체크포인트 선택은 cal-sub 300응답의 **보정 정답(R4 C1)** 으로 한다(학습 정답과 같은 기준).
4. **평가**: dev-L에서 **원래 정답과 보정 정답(C1) 둘 다**로 잰다. 임계값은 각 기준의 cal-L로 보정한다. 비교 대상은 R1b V0b(같은 방식, 원래 학습 정답)의 저장된 점수.
   비교하는 두 모델을 같은 평가 정답으로 재므로 R4의 위약 문제(정답 추가에 따른 기계적 변화)는 생기지 않는다.

## 사전 정의 기준 (결과 보기 전, SSOT §6에 고정)

- **R4b-0 재판정 점검**: R4b-A 관문 통과(Opus와 κ ≥ 0.50, 원래 환각 문장 H·A ≥ 0.70), 학습 응답 판정 확보 ≥ 98%, 학습 데이터의 원래 환각 문장 H·A ≥ 0.70. 미달이면 이후 수치는 해석 보류.
- **판정 방식**: 4토큰으로 고정(R1c 선택과 같음). 문장 끝은 보고.
- **R4b-1 보정 정답(주)**: dev-L C1에서 4토큰 **새 모델 시드 평균 − R1b V0b 시드 평균 ≥ +0.03**, 그리고 같은 시드끼리 쌍체 부트스트랩 하한 > 0이 3개 중 2개 이상.
- **R4b-2 원래 정답(비열등)**: dev-L 원래 정답에서 4토큰 새 모델 시드 평균 − R1b V0b 시드 평균 ≥ −0.02. (보정 정답에만 맞춘 것이 아님을 확인)
- **보고(판정 없음)**: 문장 끝 판정의 같은 비교, C1에서 B 문장 끝(0.621)과의 차이, 지연, 발생 전 경보, 서비스 10% 재현율, 응답·완성 문장 AUROC(두 기준), 과제별, 학습 데이터에서 바뀐 비율.
- 해석 규칙: R4b-1·R4b-2 모두 충족이면 "정답 정제가 앞부분 판정기의 성능을 올린다 → 게이트웨이 판정기 교체 후보". R4b-1 미달이면 "학습 정답 누락은 병목이 아님 → 문장 끝까지 봐야 하는 환각은 앞부분 판정으로 잡기 어려움(과제 정의 쪽)".

> **런타임: A100 권장**(L4 가능). 학습 3회 × 약 15~25분.
> **API 키**: 코랩 보안 비밀 `ANTHROPIC_API_KEY`. **크레딧 $15 이상** 필요(예상 약 $10, 3절에서 다시 계산).
> 드라이브에 `student_vark`, `r0/`, `r1b/`(V0b·V1b scores.npy), `r4/r4_corrected_labels.json`이 있어야 한다.


## 1. 환경 준비

In [ ]:
!pip -q install pysbd sentence-transformers tiktoken anthropic
!git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git 2>/dev/null || echo 'RAGTruth already cloned'

import json, time, random, os, gc, copy
import numpy as np, pandas as pd, torch, pysbd, tiktoken
from sklearn.metrics import roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

USE_DRIVE = True
OUT_DIR = '/content/drive/MyDrive/selfproject' if USE_DRIVE else '.'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
STUDENT_DIR = os.path.join(OUT_DIR, 'student_vark')
R0_DIR = os.path.join(OUT_DIR, 'r0')
R1_DIR = os.path.join(OUT_DIR, 'r4b')
R1B_DIR = os.path.join(OUT_DIR, 'r1b')
R4_DIR = os.path.join(OUT_DIR, 'r4')
os.makedirs(R1_DIR, exist_ok=True)
assert os.path.isdir(STUDENT_DIR), f'{STUDENT_DIR} 없음'
assert os.path.exists(os.path.join(R4_DIR, 'r4_corrected_labels.json')), 'R4 보정 정답(r4/r4_corrected_labels.json)이 있어야 한다'
print('device', DEVICE)

## 2. 데이터 (R1b와 같음)

In [ ]:
BASE = 'RAGTruth/dataset'
responses = [json.loads(l) for l in open(f'{BASE}/response.jsonl')]
sources   = {json.loads(l)['source_id']: json.loads(l) for l in open(f'{BASE}/source_info.jsonl')}
for r in responses:
    r['task_type'] = sources[r['source_id']]['task_type']

def build_reference(src):
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    return '\n'.join(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}"
                     for k, v in (si.items() if isinstance(si, dict) else []))

TASKS = ['QA', 'Summary', 'Data2txt']
train_pool = [r for r in responses if r['split'] == 'train']          # test 분할은 사용하지 않음
random.seed(SEED)
dev = []
for tt in TASKS:
    dev += random.sample([r for r in train_pool if r['task_type'] == tt], 120)
dev_ids = {r['id'] for r in dev}
rest = [r for r in train_pool if r['id'] not in dev_ids]
rng = random.Random(11)
teacher_resp = []
for tt in TASKS:
    teacher_resp += rng.sample([r for r in rest if r['task_type'] == tt], 200)
rng.shuffle(teacher_resp)
val = teacher_resp[:int(len(teacher_resp) * 0.15)]                   # student 학습에 쓰지 않은 검증 분할
assert not (dev_ids & {r['id'] for r in val})
print(f'dev {len(dev)} 응답 | 검증 {len(val)} 응답')

In [ ]:
# 학습 응답: S2/S2-b의 gold-large와 같은 구성 (teacher 학습 510 + 추가 2,490). dev·검증과 겹치지 않음을 확인한다.
teacher_train = teacher_resp[len(val):]
used = dev_ids | {r['id'] for r in teacher_resp}
gl_extra = rng.sample([r for r in rest if r['id'] not in used], 3000 - len(teacher_train))
train = teacher_train + gl_extra
tr_ids = {r['id'] for r in train}
assert not (tr_ids & dev_ids) and not (tr_ids & {r['id'] for r in val}), '학습과 dev/검증 겹침'
print(f'학습 {len(train)} 응답 (dev·검증과 겹침 없음)')

In [ ]:
ENC = tiktoken.get_encoding('cl100k_base')
seg = pysbd.Segmenter(language='en', clean=False, char_span=True)

def tokenize_offsets(text):
    ids = ENC.encode(text)
    dec, starts = ENC.decode_with_offsets(ids)
    if dec != text:                       # 드문 경우: 멀티바이트 문자 경계 — 누적 디코드로 대체
        starts, pos = [], 0
        for i in range(len(ids)):
            starts.append(pos); pos = len(ENC.decode(ids[:i + 1]))
    ends = starts[1:] + [len(text)]
    return starts, ends

def prep(r):
    text = r['response']
    ts, te = tokenize_offsets(text)
    # 토큰의 '내용 시작' 위치: cl100k 토큰은 앞 공백을 포함하므로(" The"), 공백을 건너뛴 위치로 문장·환각 구간에 배정한다.
    # (v1에서는 공백 위치로 배정해 문장 경계 토큰이 앞 문장에 붙는 오류가 발생 — 발생 위치의 18%)
    cs = [a + (len(text[a:b]) - len(text[a:b].lstrip())) for a, b in zip(ts, te)]
    cs = [c if c < b else a for c, a, b in zip(cs, ts, te)]          # 공백뿐인 토큰
    spans = [(l['start'], l['end']) for l in (r.get('labels') or [])]
    bad = [j for j in range(len(ts)) if any(min(te[j], b) > max(cs[j], a) for a, b in spans)]
    sents = []
    for sp in seg.segment(text):
        if not sp.sent.strip(): continue
        a, b = sp.start, sp.end
        tok = [j for j in range(len(ts)) if a <= cs[j] < b]
        if not tok: continue
        y = int(any(min(b, e) > max(a, s) for s, e in spans))
        sents.append({'a': a, 'b': b, 't0': tok[0], 't1': tok[-1], 'y': y})
    sent_of = {t: sn['t0'] for sn in sents for t in range(sn['t0'], sn['t1'] + 1)}
    return {'id': str(r['id']), 'task_type': r['task_type'], 'gen_model': r['model'],
            'ref_key': r['source_id'], 'text': text, 'ts': ts, 'te': te, 'n_tok': len(ts),
            'onset': bad[0] if bad else None, 'sents': sents, 'sent_of': sent_of,
            'ends': {sn['t1'] for sn in sents}}

REFS = {}
def to_items(rs):
    out = []
    for r in rs:
        REFS.setdefault(r['source_id'], build_reference(sources[r['source_id']]))
        out.append(prep(r))
    return out

dev_items, val_items = to_items(dev), to_items(val)
for name, it in [('dev', dev_items), ('검증', val_items)]:
    n_h = sum(x['onset'] is not None for x in it)
    print(f'{name}: 응답 {len(it)} (환각 {n_h}, 없음 {len(it)-n_h}) | 토큰 {sum(x["n_tok"] for x in it):,} | '
          f'환각 없는 응답 토큰 {sum(x["n_tok"] for x in it if x["onset"] is None):,}')

## 3. 관문 G — R4에서 받은 Sonnet·Opus 판정의 cal-L 부분으로 계산 (비용 없음)

In [ ]:
import pickle
from sklearn.metrics import cohen_kappa_score
R0P = pickle.load(open(os.path.join(R0_DIR, 'r0_items.pkl'), 'rb'))
CAL = {f'calL-{x["id"]}': x for x in R0P['items']['calL']}
load = lambda f: {k: {int(i): v for i, v in d.items()} for k, d in json.load(open(os.path.join(R4_DIR, f))).items()}
OPUS, SON = load('labels_claude-opus-5-5.json'), load('labels_claude-sonnet-5-5.json')
ids = [c for c in SON if c in CAL and c in OPUS]
rows = [(CAL[c]['sents'][i]['y'], OPUS[c][i + 1]['label'], SON[c][i + 1]['label'], CAL[c]['task_type'])
        for c in ids for i in range(len(CAL[c]['sents']))]
J = pd.DataFrame(rows, columns=['y', 'opus', 'son', 'task'])
kappa_H = cohen_kappa_score(J.opus == 'H', J.son == 'H'); sens_HA = J[J.y == 1].son.isin(['H', 'A']).mean()
GATE = kappa_H >= 0.50 and sens_HA >= 0.70
print(f'cal-L {len(ids)}응답·{len(J):,}문장 | Opus와 κ(H) {kappa_H:.3f} (≥ 0.50) | 원래 환각 문장 H·A {sens_HA:.3f} (≥ 0.70) | H 비율 Sonnet {(J.son == "H").mean():.3f} vs Opus {(J.opus == "H").mean():.3f}')
print(f"=== 관문 G: {'통과' if GATE else '미달 — 재판정하지 않음'} ===")
json.dump({'llm': 'claude-sonnet-5-5', 'gate': bool(GATE), 'kappa_H': kappa_H, 'sens_HA': float(sens_HA), 'n_resp': len(ids), 'n_sent': len(J),
           'h_rate_open': float((J.son == 'H').mean()), 'h_rate_opus': float((J.opus == 'H').mean()), 'source': 'R4 labels (cal-L part)'},
          open(os.path.join(R1_DIR, 'judge_gate.json'), 'w'), indent=1)
assert GATE, '관문 미달'

## 3-1. 학습 데이터 재판정 요청 — `claude-sonnet-5-5` 배치 (R4와 같은 지시문)

In [ ]:
import anthropic

def get_key(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    from getpass import getpass
    return getpass(f'{name}: ')

JUDGE = 'claude-sonnet-5-5'
PRICE = {'claude-sonnet-5-5': (1.0, 5.0), 'claude-opus-5-5': (2.0, 10.0)}        # 배치 가격 $/백만 토큰, 2026-10
TOK_PER_RESP = {'claude-sonnet-5-5': (1857, 280), 'claude-opus-5-5': (1821, 453)}  # R4 실측 응답당 (입력, 출력) 토큰

SYSTEM = '''You are an expert annotator for RAGTruth, a benchmark of hallucinations in retrieval-augmented generation.
You receive a REFERENCE that a language model was given, and the model's RESPONSE split into numbered sentences.
Label EVERY sentence against the REFERENCE only (not world knowledge).

A sentence contains a hallucination if any part of it is:
- Evident conflict: clearly contradicts the reference (wrong fact, number, name, date, negation).
- Subtle conflict: deviates from the reference in a way that changes meaning (altered term, scope, time, attribution).
- Evident baseless info: introduces information that is not in the reference, even if it may be true in the real world.
- Subtle baseless info: adds unsupported inference, opinion, or embellishment (e.g. "popular", "cozy", "family-friendly") that the reference does not support.

Not hallucinations: faithful paraphrase; reasonable summarization; statements that the reference lacks the information or declining to answer;
meta phrases such as "Based on the passages,". For Data2txt (business data + user reviews), claims supported by the structured data or by what reviewers say are supported.

Labels: "S" = no hallucination. "H" = contains a hallucination. "A" = ambiguous, careful annotators could reasonably disagree.
For every H or A sentence also give: "type" (one of: evident_conflict, subtle_conflict, evident_baseless, subtle_baseless),
"phrase" (the shortest span copied EXACTLY, character for character, from that sentence that is hallucinated), and "reason" (at most 20 words).

Return ONLY a JSON object, no other text:
{"sentences": [{"i": 1, "label": "S"}, {"i": 2, "label": "H", "type": "evident_baseless", "phrase": "...", "reason": "..."}]}
with exactly one entry per numbered sentence, in order.'''

TASK_DESC = {'QA': 'QA — answer the question using the passages',
             'Summary': 'Summary — summarize the news article',
             'Data2txt': 'Data2txt — write an overview of a business from structured data and user reviews'}

TRAIN_ITEMS = to_items(train)
ITEMS = {f'tr-{x["id"]}': x for x in TRAIN_ITEMS}
assert len(ITEMS) == len(TRAIN_ITEMS)

def sent_texts(x):
    return [x['text'][sn['a']:sn['b']].strip() for sn in x['sents']]

def user_msg(x):
    lines = '\n'.join(f'[{i + 1}] {s}' for i, s in enumerate(sent_texts(x)))
    return f"TASK: {TASK_DESC[x['task_type']]}\n\nREFERENCE:\n<<<\n{REFS[x['ref_key']]}\n>>>\n\nRESPONSE ({len(x['sents'])} numbered sentences):\n{lines}"

def params(cid):
    return {'model': JUDGE, 'max_tokens': 4096, 'system': SYSTEM, 'messages': [{'role': 'user', 'content': user_msg(ITEMS[cid])}]}

def parse(text, n):
    try:
        d = json.loads(text[text.find('{'): text.rfind('}') + 1])
        labs = {}
        for e in d.get('sentences', []):
            i, lab = int(e.get('i', 0)), str(e.get('label', '')).strip().upper()[:1]
            if 1 <= i <= n and lab and lab in 'SHA':
                labs[i] = {'label': lab, 'type': e.get('type', ''), 'phrase': e.get('phrase', '') or '', 'reason': e.get('reason', '')}
        return labs, len(labs) == n
    except Exception:
        return {}, False

n_resp = len(ITEMS)
tin, tout = TOK_PER_RESP[JUDGE]
cost = n_resp * (tin * PRICE[JUDGE][0] + tout * PRICE[JUDGE][1]) / 1e6
print(f'학습 응답 {n_resp:,} (환각 {sum(x["onset"] is not None for x in TRAIN_ITEMS):,}) | 문장 {sum(len(x["sents"]) for x in TRAIN_ITEMS):,}')
print(f'{JUDGE} 배치 예상 비용 약 ${cost:.1f} (R4 실측 응답당 입력 {tin}, 출력 {tout} 토큰 기준)')

### 3-1-1. 3건만 바로 보내 형식 확인 (비용 수 센트)

In [ ]:
A = anthropic.Anthropic(api_key=get_key('ANTHROPIC_API_KEY'), max_retries=2)
def call_now(cid):
    msg = A.messages.create(**params(cid))
    return ''.join(b.text for b in msg.content if getattr(b, 'type', '') == 'text'), msg.usage
for cid in list(ITEMS)[:3]:
    txt, u = call_now(cid)
    labs, ok = parse(txt, len(ITEMS[cid]['sents']))
    print(f"{cid}: 문장 {len(ITEMS[cid]['sents'])} | 파싱 {'성공' if ok else '실패'} | 입력 {u.input_tokens} / 출력 {u.output_tokens} |", {k: v['label'] for k, v in labs.items()})

### 3-2. 배치 제출(400건씩) → 결과 받기

배치 번호·결과는 드라이브 `r4b/`에 저장되고, 다시 실행하면 이미 제출한 묶음·받은 결과는 건너뛴다. 배치를 기다리는 동안 GPU는 놀고 있으므로, 길어지면 런타임을 끊었다가 나중에 1절부터 다시 실행해도 된다(재제출하지 않음).

In [ ]:
CHUNK = 400
BATCH_P, LAB_P = os.path.join(R1_DIR, 'batches.json'), os.path.join(R1_DIR, 'labels_train.json')
if os.path.exists(LAB_P):
    LABELS = {k: {int(i): v for i, v in d.items()} for k, d in json.load(open(LAB_P)).items()}
    print(f'저장된 재판정 {len(LABELS):,}건')
else:
    BATCHES = json.load(open(BATCH_P)) if os.path.exists(BATCH_P) else {'chunks': []}
    sent = {c for ch in BATCHES['chunks'] for c in ch['cids']}
    todo = [c for c in ITEMS if c not in sent]
    print(f'이미 제출 {len(sent):,} | 이번에 제출 {len(todo):,}')
    for k in range(0, len(todo), CHUNK):
        part = todo[k:k + CHUNK]
        try:
            b = A.messages.batches.create(requests=[{'custom_id': c, 'params': params(c)} for c in part])
        except Exception as e:
            print(f'  ✗ 제출 실패 ({len(part)}건): {type(e).__name__}: {str(e)[:300]}'); raise
        BATCHES['chunks'].append({'id': b.id, 'cids': part}); json.dump(BATCHES, open(BATCH_P, 'w'))
        print(f'  제출 {b.id} ({len(part)}건)', flush=True)
    LABELS, bad, tin, tout = {}, [], 0, 0
    for ch in BATCHES['chunks']:
        while True:
            b = A.messages.batches.retrieve(ch['id']); rc = b.request_counts
            print(f'  {time.strftime("%H:%M:%S")} {ch["id"][-8:]} {b.processing_status} | 처리 중 {rc.processing}, 성공 {rc.succeeded}, 오류 {rc.errored}', flush=True)
            if b.processing_status == 'ended': break
            time.sleep(60)
        for r in A.messages.batches.results(ch['id']):
            if r.result.type == 'succeeded':
                msg = r.result.message; tin += msg.usage.input_tokens; tout += msg.usage.output_tokens
                labs, ok = parse(''.join(x.text for x in msg.content if getattr(x, 'type', '') == 'text'), len(ITEMS[r.custom_id]['sents']))
                if ok: LABELS[r.custom_id] = labs; continue
            bad.append(r.custom_id)
    for cid in bad[:100]:                                    # 실패분은 바로 요청 1회 (크레딧 부족 등으로 실패하면 원래 정답 유지)
        try:
            txt, u = call_now(cid); labs, ok = parse(txt, len(ITEMS[cid]['sents']))
            if ok: LABELS[cid] = labs
        except Exception as e:
            print(f'  {cid}: {type(e).__name__} {str(e)[:100]}'); break
    USAGE = {'input_tokens': tin, 'output_tokens': tout, 'cost_usd': tin / 1e6 * PRICE[JUDGE][0] + tout / 1e6 * PRICE[JUDGE][1]}
    print(f'재판정 확보 {len(LABELS):,}/{len(ITEMS):,} | 배치 사용량 입력 {tin/1e6:.2f}M, 출력 {tout/1e6:.2f}M → 약 ${USAGE["cost_usd"]:.1f}')
    json.dump({k: {str(i): v for i, v in d.items()} for k, d in LABELS.items()}, open(LAB_P, 'w'), ensure_ascii=False)
    json.dump(USAGE, open(os.path.join(R1_DIR, 'usage_train.json'), 'w'))

In [ ]:
GATE_META = json.load(open(os.path.join(R1_DIR, 'judge_gate.json')))
print(f"재판정자 {GATE_META['llm']} | 관문: κ {GATE_META['kappa_H']:.3f}, H·A {GATE_META['sens_HA']:.3f} → {'통과' if GATE_META['gate'] else '미달'} | 학습 판정 {len(LABELS):,}/{len(ITEMS):,}")

### 3-3. R4b-0 점검과 보정 학습 정답

In [ ]:
coverage = len(LABELS) / len(ITEMS)
pos_lab = [LABELS[c][i + 1]['label'] for c, x in ITEMS.items() if c in LABELS for i, sn in enumerate(x['sents']) if sn['y'] == 1]
neg_lab = [LABELS[c][i + 1]['label'] for c, x in ITEMS.items() if c in LABELS for i, sn in enumerate(x['sents']) if sn['y'] == 0]
sens_HA = float(np.mean([l in 'HA' for l in pos_lab]))
R4B_0 = bool(GATE_META['gate']) and coverage >= 0.98 and sens_HA >= 0.70
print(f"관문 {'통과' if GATE_META['gate'] else '미달'} | 확보율 {coverage:.3f} (≥ 0.98)" + f' | 원래 환각 문장 {len(pos_lab):,}개에 H·A {sens_HA:.3f} (≥ 0.70) | 원래 멀쩡한 문장 {len(neg_lab):,}개에 H {np.mean([l == "H" for l in neg_lab]):.3f}')
print(f"=== R4b-0 재판정 점검: {'충족' if R4B_0 else '미달 — 이후 수치는 해석 보류'} ===")

def content_starts(x):
    t = x['text']
    cs = [a + (len(t[a:b]) - len(t[a:b].lstrip())) for a, b in zip(x['ts'], x['te'])]
    return [c if c < b else a for c, a, b in zip(cs, x['ts'], x['te'])]

def locate(x, sn, phrase, cs):
    seg = x['text'][sn['a']:sn['b']]
    ph = (phrase or '').strip()
    k = seg.find(ph) if ph else -1
    if k < 0 and ph: k = seg.lower().find(ph.lower())
    if k < 0: return sn['t0']
    a, b = sn['a'] + k, sn['a'] + k + len(ph)
    for t in range(sn['t0'], sn['t1'] + 1):
        if min(x['te'][t], b) > max(cs[t], a): return t
    return sn['t0']

ADD_ONSET, flip, earlier = {}, 0, 0
for cid, x in ITEMS.items():
    labs = LABELS.get(cid)
    if not labs: continue
    cs, add = None, []
    for i, sn in enumerate(x['sents']):
        if sn['y'] == 0 and labs[i + 1]['label'] == 'H':
            cs = cs or content_starts(x)
            t = locate(x, sn, labs[i + 1]['phrase'], cs); ADD_ONSET[(x['id'], i)] = t; add.append(t)
    if add:
        if x['onset'] is None: flip += 1
        elif min(add) < x['onset']: earlier += 1
n_clean = sum(x['onset'] is None for x in TRAIN_ITEMS); n_hall = len(TRAIN_ITEMS) - n_clean
print(f'추가된 환각 문장 {len(ADD_ONSET):,} | 멀쩡 → 환각 {flip}/{n_clean} ({flip / max(n_clean, 1):.0%}) | 더 이른 환각 {earlier}/{n_hall} ({earlier / max(n_hall, 1):.0%})')

## 4. 학습 판정 지점 — 보정 학습 정답으로 (R1b V0b와 같은 정의)

In [ ]:
def points(item, s):
    pts = []
    for sn in item['sents']:
        for t in range(sn['t0'], sn['t1'] + 1):
            k = t - sn['t0'] + 1
            if (s is not None and k % s == 0) or t == sn['t1']:
                pts.append((t, item['text'][sn['a']:min(item['te'][t], sn['b'])].strip()))
    return pts


WIN_TOK = 10
train_items = TRAIN_ITEMS
by_tr = {str(r['id']): r for r in train}
TR, P_KEY, P_TXT, P_CAT = [], [], [], []
for x in train_items:
    text = x['text']
    spans = [(l['start'], l['end']) for l in (by_tr[x['id']].get('labels') or [])]
    cs = [a + (len(text[a:b]) - len(text[a:b].lstrip())) if text[a:b].strip() else a for a, b in zip(x['ts'], x['te'])]
    def s_onset(sn):                      # E2와 같다: 이 문장 안에서 환각 구간과 처음 겹치는 토큰
        for t in range(sn['t0'], sn['t1'] + 1):
            if any(min(x['te'][t], b) > max(cs[t], a) for a, b in spans): return t
        return None
    sid = {sn['t0']: i for i, sn in enumerate(x['sents'])}
    son = [s_onset(sn) for sn in x['sents']]
    son = [o if o is not None else ADD_ONSET.get((x['id'], i)) for i, o in enumerate(son)]   # R4b: 재판정이 추가한 환각 문장
    groups = {}
    for t, txt in points(x, 2):
        if not txt: continue
        si = sid[x['sent_of'][t]]; o = son[si]
        if o is None: cat = 'neg'
        elif t < o: cat = 'pre'
        elif t <= o + WIN_TOK: cat = 'win'
        else: cat = 'post'
        g = len(P_TXT); P_KEY.append(x['ref_key']); P_TXT.append(txt); P_CAT.append(cat)
        if cat in ('win', 'post'): groups.setdefault(si, []).append(g)
    gl = []
    for si, lst in groups.items():
        win = [g for g in lst if P_CAT[g] == 'win']
        if not win:                       # 창 안에 지점이 없으면 시작 이후 첫 지점을 창으로
            P_CAT[lst[0]] = 'win'; win = [lst[0]]
        gl.append(win)
    TR.append({'groups': gl})
P_CAT = np.array(P_CAT)
print(f'학습 응답 {len(TR)} | 판정 지점 {len(P_TXT):,} | 환각 문장(다중 사례 단위) {sum(len(r["groups"]) for r in TR):,} |',
      pd.Series(P_CAT).value_counts().to_dict())
assert sum(len(r['groups']) for r in TR) > 0

## 5. 근거 검색 (R1b와 같음)과 평가용 보정 정답(R4 C1)

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

retriever = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEVICE)
K, MAX_LEN, CHUNK_WORDS, BS = 8, 512, 40, 32

chunk_cache = {}
def get_chunks(key):
    if key not in chunk_cache:
        w = REFS[key].split()
        ch = [' '.join(w[i:i + CHUNK_WORDS]) for i in range(0, max(len(w), 1), CHUNK_WORDS)] or [REFS[key]]
        chunk_cache[key] = (ch, retriever.encode(ch, convert_to_tensor=True, normalize_embeddings=True,
                                                 batch_size=128, show_progress_bar=False))
    return chunk_cache[key]

def contexts(keys, texts, bs=256):
    # 텍스트마다 근거 top-K 조각을 원문 순서로 이어 붙인 문자열
    out = []
    for i in range(0, len(texts), bs):
        embs = retriever.encode(texts[i:i + bs], convert_to_tensor=True, normalize_embeddings=True,
                                batch_size=bs, show_progress_bar=False)
        for key, e in zip(keys[i:i + bs], embs):
            ch, ce = get_chunks(key)
            top = torch.topk(ce @ e, min(K, ce.shape[0])).indices.tolist()
            out.append(' ... '.join(ch[t] for t in sorted(top)))
    return out


# R0 평가 세트(cal-L, dev-L)와 기존 판정기 점수
import pickle
R0P = pickle.load(open(os.path.join(R0_DIR, 'r0_items.pkl'), 'rb'))
Z0 = dict(np.load(os.path.join(R0_DIR, 'r0_scores.npz')))
CAL, DEV = R0P['items']['calL'], R0P['items']['devL']
Q_KEYS, Q_TEXTS = R0P['q_keys'], R0P['q_texts']
for k in set(Q_KEYS):
    REFS.setdefault(k, build_reference(sources[k]))

@torch.no_grad()
def topk_idx(keys, texts, bs=512):
    out = []
    for i in range(0, len(texts), bs):
        embs = retriever.encode(texts[i:i + bs], convert_to_tensor=True, normalize_embeddings=True,
                                batch_size=bs, show_progress_bar=False)
        for key, e in zip(keys[i:i + bs], embs):
            ch, ce = get_chunks(key)
            out.append(sorted(torch.topk(ce @ e, min(K, ce.shape[0])).indices.tolist()))
    return out
ctx_of = lambda key, idx: ' ... '.join(get_chunks(key)[0][j] for j in idx)

t0 = time.perf_counter()
P_IDX = topk_idx(P_KEY, P_TXT)
print(f'학습 지점 근거 검색 {time.perf_counter()-t0:.0f}초')

# 평가 질의: cal-L·dev-L 지점만 (R0와 같은 질의 번호)
EVAL_Q = sorted({q for x in CAL + DEV for _, q in x['pts2']})
t0 = time.perf_counter()
EQ_IDX = dict(zip(EVAL_Q, topk_idx([Q_KEYS[q] for q in EVAL_Q], [Q_TEXTS[q] for q in EVAL_Q])))
print(f'평가 질의 {len(EVAL_Q):,}개 근거 검색 {time.perf_counter()-t0:.0f}초')

# 체크포인트 선택용 cal-sub (cal-L에서 300응답 고정)
CALSUB = random.Random(1).sample(CAL, min(300, len(CAL)))

In [ ]:
# R4 보정 정답 C1 (Opus 재판정)을 cal-L·dev-L에 적용한 사본
CORR = json.load(open(os.path.join(R4_DIR, 'r4_corrected_labels.json')))['C1']
def apply_corr(items, split):
    out = []
    for x in items:
        c = CORR[split].get(x['id'])
        y = copy.copy(x); y['sents'] = [dict(sn) for sn in x['sents']]
        if c is not None:
            y['onset'] = c['onset']
            for i in c['added']: y['sents'][i - 1]['y'] = 1
        out.append(y)
    return out
CAL_C, DEV_C = apply_corr(CAL, 'calL'), apply_corr(DEV, 'devL')
LAB_SETS = {'orig': (CAL, DEV), 'C1': (CAL_C, DEV_C)}
for k, (c, d) in LAB_SETS.items():
    print(f"{k}: cal-L 환각 {sum(x['onset'] is not None for x in c)}, dev-L 환각 {sum(x['onset'] is not None for x in d)}")
# 체크포인트 선택은 보정 정답 기준 cal-sub (같은 300응답)
_byid = {x['id']: x for x in CAL_C}
CALSUB = [_byid[x['id']] for x in CALSUB]

## 6. 평가 함수 (R1b와 같음)

In [ ]:
SC = dict(Z0)

def series(x, scorer):
    m, k = scorer.split('|')
    S_ = SC[m]
    if k == 'end':
        return [(t, S_[q]) for t, q in x['pts2'] if t in x['ends']]
    if k == 'ew2':
        out, s = [], None
        for t, q in x['pts2']:
            v = float(S_[q]); s = v if s is None else 0.5 * v + 0.5 * s
            out.append((t, s))
        return out
    s = int(k)
    return [(t, S_[q]) for t, q in x['pts2'] if t in x['ends'] or (t - x['sent_of'][t] + 1) % s == 0]

In [ ]:
def calibrate(items, scorer, arl0):
    clean = [x for x in items if x['onset'] is None]
    n_tok = sum(x['n_tok'] for x in clean)
    sc = np.sort(np.concatenate([[s for _, s in series(x, scorer)] for x in clean]))[::-1]
    n_alarm = max(1, int(n_tok / arl0))
    return float(sc[n_alarm - 1]) + 1e-6 if n_alarm <= len(sc) else float(sc[-1])

def evaluate(items, scorer, thr):
    # 엄격 기준(주 지표, 선행 연구와 같은 정의): 응답의 **첫 경보**가 발생 θ 이후여야 탐지로 인정.
    #   θ 전에 경보가 울리면 그 응답은 오경보로 끝난 것(미탐지)으로 본다.
    # 관대 기준(참고): θ 전 경보는 무시하고 θ 이후 첫 경보를 탐지로 인정 — v1의 주 지표였으나 재현율이 부풀려짐.
    clean = [x for x in items if x['onset'] is None]
    hall = [x for x in items if x['onset'] is not None]
    alarms = sum(sum(s >= thr for _, s in series(x, scorer)) for x in clean)
    arl = sum(x['n_tok'] for x in clean) / max(alarms, 1)
    clean_flag = np.mean([any(s >= thr for _, s in series(x, scorer)) for x in clean])
    st_det, st_d, st_ch, cedd, le_det, pre = 0, [], [], [], 0, 0
    for x in hall:
        ser = series(x, scorer)
        first = next((t for t, s in ser if s >= thr), None)
        le_hit = next((t for t, s in ser if t >= x['onset'] and s >= thr), None)
        le_det += le_hit is not None
        pre += first is not None and first < x['onset']
        if first is not None and first >= x['onset']:
            st_det += 1; st_d.append(first - x['onset']); cedd.append(first - x['onset'])
            st_ch.append(x['te'][first] - x['ts'][x['onset']])
        else:
            cedd.append(x['n_tok'] - x['onset'])
    return {'realized_ARL0_dev': arl, 'clean_resp_flagged': clean_flag,
            'recall': st_det / len(hall), 'delay_tok_mean': np.mean(st_d) if st_d else np.nan,
            'delay_tok_median': np.median(st_d) if st_d else np.nan,
            'delay_chars_median': np.median(st_ch) if st_ch else np.nan,
            'censored_EDD_tok': np.mean(cedd), 'pre_onset_alarm_rate': pre / len(hall),
            'recall_lenient': le_det / len(hall),
            'calls_per_resp': np.mean([len(series(x, scorer)) for x in items])}

def calibrate_flag(items, scorer, rate):
    # 서비스 기준 보정: 환각 없는 응답 중 표시되는 비율이 rate 이하가 되는 임계값 (응답별 최대 점수의 분위수)
    mx = np.array([max([s for _, s in series(x, scorer)] or [-1e9]) for x in items if x['onset'] is None])
    return float(np.quantile(mx, 1 - rate)) + 1e-6

In [ ]:
def resp_auc(items, m):
    y = [int(x['onset'] is not None) for x in items]
    s = [max([SC[m][q] for _, q in x['pts2']] or [-1e9]) for x in items]
    return roc_auc_score(y, s)

def sent_auc(items, m):
    ys, ss = [], []
    for x in items:
        d = dict(x['pts2'])
        for sn in x['sents']:
            if sn['t1'] in d: ys.append(sn['y']); ss.append(SC[m][d[sn['t1']]])
    return roc_auc_score(ys, ss)

def per_resp(items, sc, thr):
    d, l = [], []
    for x in items:
        if x['onset'] is None: continue
        first = next((t for t, s in series(x, sc) if s >= thr), None)
        ok = first is not None and first >= x['onset']
        d.append(float(ok)); l.append(first - x['onset'] if ok else np.nan)
    return np.array(d), np.array(l)

def boot(a, b, n=1000, seed=0):
    da, la = per_resp(DEV, a, calibrate(CAL, a, 100)); db, lb = per_resp(DEV, b, calibrate(CAL, b, 100))
    rng = np.random.default_rng(seed); N = len(da); rd = []
    for _ in range(n):
        i = rng.integers(0, N, N); rd.append(da[i].mean() - db[i].mean())
    lo, hi = np.percentile(rd, [2.5, 97.5])
    return {'A': a, 'B': b, 'recall_A': da.mean(), 'recall_B': db.mean(), 'diff': da.mean() - db.mean(), 'lo': lo, 'hi': hi}

# 기준선 확인 (R0 dev-L과 같아야 함)
for sc, ref in [('B|end', 0.5007), ('B|4', 0.4809), ('E2|4', 0.4506)]:
    got = evaluate(DEV, sc, calibrate(CAL, sc, 100))['recall']
    print(f'{sc:6} R0 {ref:.4f} | 지금 {got:.4f} | {"일치" if abs(got-ref) <= 0.002 else "불일치 — 확인 필요"}')

## 7. 학습 — R1b V0b와 같은 방식, 학습 정답만 보정 (끝난 것은 건너뜀)

In [ ]:
from transformers import get_linear_schedule_with_warmup
import torch.nn.functional as F

BATCH_SEQ, LR, POST_W, GROUP_W, MARGIN, MINE_M, N_EVALS = 64, 2e-5, 0.2, 5.0, 1.0, 4, 4
BF16 = DEVICE == 'cuda' and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16

def encode(tok, idxs):
    ctx = [ctx_of(P_KEY[g], P_IDX[g]) for g in idxs]
    return tok(ctx, [P_TXT[g] for g in idxs], truncation='only_first', max_length=MAX_LEN,
               padding=True, return_tensors='pt').to(DEVICE)

def make_units(variant, rng):
    units = []
    for g in range(len(P_TXT)):
        c = P_CAT[g]
        if variant == 'V0b':
            units.append(('s', g, 1.0 if c in ('win', 'post') else 0.0, 1.0))
        elif c == 'post':
            units.append(('s', g, 1.0, POST_W))
        elif c in ('neg', 'pre'):
            units.append(('s', g, 0.0, 1.0))
    if variant == 'V1b':
        for r in TR:
            for win in r['groups']:
                units.append(('g', win))
    rng.shuffle(units)
    return units

@torch.no_grad()
def score_ids(tok, model, idxs, bs=256):
    model.eval(); out = []
    for i in range(0, len(idxs), bs):
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
            out.append(model(**encode(tok, idxs[i:i + bs])).logits.squeeze(-1).float().cpu())
    model.train()
    return torch.cat(out).numpy() if out else np.zeros(0)

@torch.no_grad()
def score_queries(tok, model, qs, bs=256):
    model.eval(); out = np.zeros(len(qs), dtype=np.float32)
    order = sorted(range(len(qs)), key=lambda i: len(Q_TEXTS[qs[i]]))
    for i in range(0, len(order), bs):
        b = [qs[j] for j in order[i:i + bs]]
        ctx = [ctx_of(Q_KEYS[q], EQ_IDX[q]) for q in b]
        enc = tok(ctx, [Q_TEXTS[q] for q in b], truncation='only_first', max_length=MAX_LEN,
                  padding=True, return_tensors='pt').to(DEVICE)
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
            out[order[i:i + bs]] = model(**enc).logits.squeeze(-1).float().cpu().numpy()
    model.train()
    return out

CALSUB_Q = sorted({q for x in CALSUB for _, q in x['pts2']})
def calsub_metric(tok, model):
    v = score_queries(tok, model, CALSUB_Q)
    SC['_cur'] = np.zeros(len(Q_KEYS), dtype=np.float32); SC['_cur'][CALSUB_Q] = v
    r4 = evaluate(CALSUB, '_cur|4', calibrate(CALSUB, '_cur|4', 100))
    r2 = evaluate(CALSUB, '_cur|2', calibrate(CALSUB, '_cur|2', 100))
    d4 = r4['delay_tok_median']
    return float((r4['recall'] + r2['recall']) / 2), float(d4) if d4 == d4 else 1e9

def group_loss(variant, win, tok, model):
    return win, []

def train_variant(variant, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tok = AutoTokenizer.from_pretrained(STUDENT_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(STUDENT_DIR).to(DEVICE)
    units = make_units(variant, random.Random(seed))
    n_seq = sum(1 if u[0] == 's' else 6 for u in units)
    n_steps = max(1, n_seq // BATCH_SEQ)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * n_steps), n_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda' and not BF16))
    best, d0 = calsub_metric(tok, model)
    best_key, best_state, hist = (d0 <= 9, best), copy.deepcopy(model.state_dict()), [(0, best, d0)]
    print(f'  [{variant} s{seed}] 시작 cal-sub 재현율 {best:.3f}, 지연 {d0} | 예상 {n_steps}단계')
    eval_at = {int(n_steps * (i + 1) / N_EVALS) for i in range(N_EVALS)}
    model.train(); step, ui, t0 = 0, 0, time.perf_counter()
    while ui < len(units):
        singles, groups, n = [], [], 0
        while ui < len(units) and n < BATCH_SEQ:
            u = units[ui]; ui += 1
            if u[0] == 's':
                singles.append(u); n += 1
            else:
                win, hard = group_loss(variant, u[1], tok, model)
                if win:
                    groups.append((win, hard)); n += len(win) + len(hard)
        ids = [u[1] for u in singles] + [g for w, h in groups for g in w + h]
        if not ids: continue
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
            z = model(**encode(tok, ids)).logits.squeeze(-1).float()
        ns = len(singles)
        y = torch.tensor([u[2] for u in singles], device=DEVICE); w = torch.tensor([u[3] for u in singles], device=DEVICE)
        loss_num = (F.binary_cross_entropy_with_logits(z[:ns], y, reduction='none') * w).sum() if ns else z.sum() * 0
        den = w.sum() if ns else torch.tensor(0.0, device=DEVICE)
        pos = ns
        for win, hard in groups:
            zw = z[pos:pos + len(win)]; zh = z[pos + len(win):pos + len(win) + len(hard)]; pos += len(win) + len(hard)
            s_no = F.logsigmoid(-zw).sum().clamp(max=-1e-6)              # 창 안에서 한 번도 안 울릴 로그확률
            l = -torch.log(-torch.expm1(s_no))                            # 적어도 하나 울릴 확률의 음의 로그
            if len(hard):
                l = l + F.softplus(torch.logsumexp(zh, 0) - torch.logsumexp(zw, 0) + MARGIN)
            loss_num = loss_num + GROUP_W * l; den = den + GROUP_W
        loss = loss_num / den.clamp(min=1e-6)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step()
        step += 1
        if step in eval_at or ui >= len(units):
            va, dv = calsub_metric(tok, model); hist.append((step, va, dv))
            if (dv <= 9, va) > best_key: best_key, best, best_state = (dv <= 9, va), va, copy.deepcopy(model.state_dict())
            print(f'  [{variant} s{seed}] 단계 {step} | 손실 {loss.item():.3f} | cal-sub 재현율 {va:.3f}, 지연 {dv} | {(time.perf_counter()-t0)/60:.1f}분')
    model.load_state_dict(best_state)
    return tok, model, best, hist

import shutil
def run(variant, seed):
    out = os.path.join(R1_DIR, f'{variant}_s{seed}')
    sp = os.path.join(out, 'scores.npy')
    if not os.path.exists(sp):
        has_w = any(os.path.exists(os.path.join(out, f)) for f in ['model.safetensors', 'pytorch_model.bin'])
        if os.path.exists(os.path.join(out, 'config.json')) and not has_w:
            print(f'  [{variant} s{seed}] 저장이 끊긴 폴더(가중치 없음) → 지우고 다시 학습'); shutil.rmtree(out)
        if has_w:
            tok = AutoTokenizer.from_pretrained(out); model = AutoModelForSequenceClassification.from_pretrained(out).to(DEVICE)
        else:
            tok, model, best, hist = train_variant(variant, seed)
            os.makedirs(out, exist_ok=True); tok.save_pretrained(out); model.save_pretrained(out)
            pd.DataFrame(hist, columns=['step', 'calsub_recall', 'calsub_delay4']).to_csv(os.path.join(out, 'hist.csv'), index=False)
        t0 = time.perf_counter()
        v = score_queries(tok, model, EVAL_Q)
        full = np.full(len(Q_KEYS), np.nan, dtype=np.float32); full[EVAL_Q] = v
        np.save(sp, full)
        print(f'  [{variant} s{seed}] 평가 질의 채점 {time.perf_counter()-t0:.0f}초')
        del model; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
    SC[f'{variant}_s{seed}'] = np.load(sp)
    print(f'[{variant} s{seed}] 준비 완료')

In [ ]:
SEEDS = [42, 7, 2026]
for seed in SEEDS:
    run('V0b', seed)                      # 드라이브 r4b/V0b_s*  (SC 키 'V0b_s*' = 새 모델)
for v in ['V0b', 'V1b']:
    for s in SEEDS:
        p = os.path.join(R1B_DIR, f'{v}_s{s}', 'scores.npy')
        if os.path.exists(p): SC[f'R1b{v}_s{s}'] = np.load(p)
assert all(f'R1bV0b_s{s}' in SC for s in SEEDS), 'R1b V0b 점수가 드라이브 r1b/에 있어야 한다'
print('준비된 점수:', [k for k in SC if k[-2:] in ('42', '_7') or k.endswith('2026')])

## 8. dev-L 평가 (원래 정답·보정 정답)와 사전 기준 판정

In [ ]:
KINDS = ['end', '4', '2', 'ew2']
MODELS = ['B', 'E2'] + [f'R1b{v}_s{s}' for v in ['V0b', 'V1b'] for s in SEEDS] + [f'V0b_s{s}' for s in SEEDS]
rows = []
for lab, (cal_, dev_) in LAB_SETS.items():
    for m in MODELS:
        ra, sa = resp_auc(dev_, m), sent_auc(dev_, m)
        for k in KINDS:
            sc = f'{m}|{k}'
            r = evaluate(dev_, sc, calibrate(cal_, sc, 100)); r10 = evaluate(dev_, sc, calibrate_flag(cal_, sc, 0.10))
            rows.append({'labels': lab, 'model': m, 'kind': k, 'recall': r['recall'], 'delay_med': r['delay_tok_median'],
                         'pre_onset': r['pre_onset_alarm_rate'], 'svc_recall': r10['recall'], 'svc_delay': r10['delay_tok_median'],
                         'resp_auc': ra, 'sent_auc': sa})
res = pd.DataFrame(rows)

def fam(fm, k, lab, col='recall'):
    ms = [f'{fm}_s{s}' for s in SEEDS] if fm not in ('B', 'E2') else [fm]
    return float(np.mean([res[(res.labels == lab) & (res.model == m) & (res.kind == k)][col].iloc[0] for m in ms]))

agg = pd.DataFrame([{'labels': lab, 'family': f, 'kind': k, **{c: fam(f, k, lab, c) for c in
                     ['recall', 'delay_med', 'pre_onset', 'svc_recall', 'resp_auc', 'sent_auc']}}
                    for lab in LAB_SETS for f in ['B', 'E2', 'R1bV0b', 'R1bV1b', 'V0b'] for k in KINDS])
pd.set_option('display.width', 250)
print(agg[agg.kind.isin(['4', 'end'])].round(3).to_string(index=False))

def per_resp(items, sc, thr):
    d = []
    for x in items:
        if x['onset'] is None: continue
        first = next((t for t, s in series(x, sc) if s >= thr), None)
        d.append(float(first is not None and first >= x['onset']))
    return np.array(d)

def boot_pair(a, b, lab, n=1000, seed=0):
    cal_, dev_ = LAB_SETS[lab]
    da, db = per_resp(dev_, a, calibrate(cal_, a, 100)), per_resp(dev_, b, calibrate(cal_, b, 100))
    rng = np.random.default_rng(seed); N = len(da)
    rd = [da[i].mean() - db[i].mean() for i in (rng.integers(0, N, N) for _ in range(n))]
    lo, hi = np.percentile(rd, [2.5, 97.5])
    return {'A': a, 'B': b, 'labels': lab, 'diff': da.mean() - db.mean(), 'lo': lo, 'hi': hi}

K = '4'
g_c = fam('V0b', K, 'C1') - fam('R1bV0b', K, 'C1')
bts = [boot_pair(f'V0b_s{s}|{K}', f'R1bV0b_s{s}|{K}', 'C1') for s in SEEDS]
n_sig = sum(b['lo'] > 0 for b in bts)
R4B_1 = g_c >= 0.03 and n_sig >= 2
g_o = fam('V0b', K, 'orig') - fam('R1bV0b', K, 'orig')
R4B_2 = g_o >= -0.02
print(f"\n=== R4b-1 보정 정답(4토큰): 새 {fam('V0b', K, 'C1'):.3f} − R1b V0b {fam('R1bV0b', K, 'C1'):.3f} = {g_c:+.3f} (≥ +0.03), 유의 시드 {n_sig}/3 (≥ 2) → {'충족' if R4B_1 else '미달'} ===")
for b in bts:
    print(f"   {b['A']} vs {b['B']}: {b['diff']:+.3f} [{b['lo']:+.3f}, {b['hi']:+.3f}]")
print(f"=== R4b-2 원래 정답(4토큰): 새 {fam('V0b', K, 'orig'):.3f} − R1b V0b {fam('R1bV0b', K, 'orig'):.3f} = {g_o:+.3f} (≥ −0.02) → {'충족' if R4B_2 else '미달'} ===")
if not R4B_0:
    print('※ R4b-0 미달: 위 판정은 해석 보류')
print(f"\n(보고) 문장 끝: C1 새 {fam('V0b', 'end', 'C1'):.3f} vs R1b V0b {fam('R1bV0b', 'end', 'C1'):.3f} | 원래 새 {fam('V0b', 'end', 'orig'):.3f} vs {fam('R1bV0b', 'end', 'orig'):.3f}")
print(f"(보고) C1에서 B 문장 끝 {fam('B', 'end', 'C1'):.3f} 대비 새 4토큰 {fam('V0b', K, 'C1'):.3f} (차이 {fam('V0b', K, 'C1') - fam('B', 'end', 'C1'):+.3f}), 지연 {fam('V0b', K, 'C1', 'delay_med'):.1f} vs {fam('B', 'end', 'C1', 'delay_med'):.1f}")
print(f"(보고) 완성 문장 AUROC C1: 새 {fam('V0b', K, 'C1', 'sent_auc'):.3f} vs R1b V0b {fam('R1bV0b', K, 'C1', 'sent_auc'):.3f} | 원래: {fam('V0b', K, 'orig', 'sent_auc'):.3f} vs {fam('R1bV0b', K, 'orig', 'sent_auc'):.3f}")
print(f"(보고) 서비스 10% 재현율 C1: 새 {fam('V0b', K, 'C1', 'svc_recall'):.3f} vs R1b V0b {fam('R1bV0b', K, 'C1', 'svc_recall'):.3f}")

## 9. 과제별 (보고)

In [ ]:
tr = []
for lab, (cal_, dev_) in LAB_SETS.items():
    for tt in TASKS:
        its = [x for x in dev_ if x['task_type'] == tt]
        row = {'labels': lab, 'task': tt, 'n_hall': sum(x['onset'] is not None for x in its)}
        for sc in ['B|end'] + [f'R1bV0b_s{s}|4' for s in SEEDS] + [f'V0b_s{s}|4' for s in SEEDS]:
            row[sc] = evaluate(its, sc, calibrate(cal_, sc, 100))['recall']
        row['R1bV0b|4 평균'] = np.mean([row[f'R1bV0b_s{s}|4'] for s in SEEDS]); row['새 V0b|4 평균'] = np.mean([row[f'V0b_s{s}|4'] for s in SEEDS])
        tr.append(row)
task_df = pd.DataFrame(tr)
print(task_df[['labels', 'task', 'n_hall', 'B|end', 'R1bV0b|4 평균', '새 V0b|4 평균']].round(3).to_string(index=False))

## 10. 저장

In [ ]:
import shutil
summ = {'R4b_0': bool(R4B_0), 'coverage': coverage, 'sens_HA_train': sens_HA, 'added_sent_train': len(ADD_ONSET), 'flip_clean_train': flip,
        'earlier_train': earlier, 'R4b_1': bool(R4B_1), 'gain_C1': g_c, 'n_sig_C1': int(n_sig), 'boot_C1': bts,
        'R4b_2': bool(R4B_2), 'gain_orig': g_o, 'judge': JUDGE}
json.dump(summ, open('r4b_summary.json', 'w'), default=float, ensure_ascii=False, indent=1)
hists = []
for s in SEEDS:
    p = os.path.join(R1_DIR, f'V0b_s{s}', 'hist.csv')
    if os.path.exists(p):
        h = pd.read_csv(p); h['run'] = f'V0b_s{s}'; hists.append(h)
out = {'r4b_devL.csv': res, 'r4b_family.csv': agg, 'r4b_task.csv': task_df, 'r4b_hist.csv': pd.concat(hists) if hists else pd.DataFrame()}
for f, d in out.items():
    d.to_csv(f, index=False); d.to_csv(os.path.join(R1_DIR, f), index=False)
shutil.copy('r4b_summary.json', os.path.join(R1_DIR, 'r4b_summary.json'))
print('저장 완료: 드라이브 selfproject/r4b/')